In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from tinyshift.association_mining import TransactionAnalyzer
rng = np.random.default_rng(42)
from typing import List
import kagglehub
from kagglehub import KaggleDatasetAdapter
from doubleml import DoubleMLData, DoubleMLPLR
from lightgbm import LGBMRegressor

/home/heylucasleao/forecasting/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df_groceries = kagglehub.dataset_load(KaggleDatasetAdapter.PANDAS, "irfanasrullah/groceries", "groceries - groceries.csv")
df_groceries = df_groceries[df_groceries.columns[1:]].fillna("")
groceries = df_groceries.to_numpy()
groceries = [arr[np.where(arr != "")] for arr in groceries]

In [9]:
from typing import List, Dict, Optional, Union
import pandas as pd
import numpy as np
from scipy import sparse
from doubleml import DoubleMLData, DoubleMLPLR
from lightgbm import LGBMRegressor
from mlforecast import MLForecast


class ForecastPrice:
    """
    Pipeline de previsão de demanda com ajuste causal de elasticidade cruzada e canibalização.
    """

    def __init__(self, mlforecast_model: Optional[MLForecast] = None):
        self.mask_: Optional[pd.DataFrame] = None
        self.sparse_mask_: Optional[sparse.csr_matrix] = None
        self.competitors_dict_: Dict[str, List[str]] = {}
        self.skus_: Optional[List[str]] = None
        self.id_col_: Optional[str] = None
        self.time_col_: Optional[str] = None
        self.target_col_: Optional[str] = None
        
        # Dicionário de modelos DML por SKU alvo
        self.dml_models_: Dict[str, DoubleMLPLR] = {}
        
        # Dicionário de elasticidades cruzadas médias -> {sku_target: {sku_competitor: elasticity}}
        self.cross_elasticities_: Dict[str, Dict[str, float]] = {}
        
        # Modelo base de séries temporais (Nixtla MLForecast)
        self.mlforecast = mlforecast_model

    def fit_mask_(self, transactions: List[List[str]]):
        """
        Calcula e armazena a matriz de concorrência como DataFrame esparso.
        """
        ta = TransactionAnalyzer().fit(transactions)
        self.skus_ = list(ta.columns_)
        
        lift = ta.correlation_matrix(self.skus_, self.skus_, metric="lift")
        hypergeom_p = ta.correlation_matrix(self.skus_, self.skus_, metric="hypergeom")
        
        bool_mask = (hypergeom_p < 0.05) & (lift > 1.2)
        mask_matrix = sparse.csr_matrix(bool_mask.to_numpy(dtype=bool))
        self.sparse_mask_ = mask_matrix
        self.mask_ = pd.DataFrame.sparse.from_spmatrix(
            mask_matrix,
            index=self.skus_,
            columns=self.skus_
        )

        self.competitors_dict_ = {}
        for sku in self.skus_:
            competing = self.mask_.loc[sku][self.mask_.loc[sku]].index.tolist()
            if competing:
                self.competitors_dict_[sku] = competing

        return self

    def get_competitors(self, sku_target: str) -> List[str]:
        """
        Retorna a lista de SKUs concorrentes para um SKU específico.
        """
        return self.competitors_dict_.get(sku_target, [])

    def fit_cross_elasticity_(
        self,
        df_panel: pd.DataFrame,
        id_col: str,
        target_col: str,
        sku_target: str,
        control_cols: List[str]
    ):
        """
        Ajusta um DoubleMLPLR para estimar a elasticidade cruzada do SKU alvo.
        """
        competitors = self.get_competitors(sku_target)
        if not competitors:
            print(f"Nenhum concorrente significativo encontrado para {sku_target}.")
            return

        df_sku = df_panel[df_panel[id_col] == sku_target].copy()
        available_competitors = [
            comp for comp in competitors if f"price_{comp}" in df_sku.columns
        ]
        if not available_competitors:
            return

        treatment_cols = [f"price_{comp}" for comp in available_competitors]
        required_cols = [target_col] + treatment_cols + control_cols
        df_dml = df_sku[required_cols].dropna().reset_index(drop=True)
        if df_dml.empty:
            return

        dml_data = DoubleMLData(
            df_dml,
            y_col=target_col,
            d_cols=treatment_cols,
            x_cols=control_cols,
        )
        dml = DoubleMLPLR(
            dml_data,
            ml_l=LGBMRegressor(
                n_estimators=100,
                max_depth=3,
                verbose=-1,
                random_state=42,
            ),
            ml_m=LGBMRegressor(
                n_estimators=100,
                max_depth=3,
                verbose=-1,
                random_state=42,
            ),
            n_folds=3,
        )
        dml.fit()

        self.dml_models_[sku_target] = dml
        self.cross_elasticities_[sku_target] = {
            comp: float(dml.summary.loc[f"price_{comp}", "coef"])
            for comp in available_competitors
        }

    def fit_mlforecast_(
        self,
        df_panel: pd.DataFrame,
        id_col: str,
        time_col: str,
        target_col: str,
        static_features: Optional[List[str]] = None,
    ):
        """
        Treina o modelo de séries temporais da Nixtla (MLForecast).
        """
        if self.mlforecast is None:
            raise ValueError("Forneça uma instância válida de MLForecast no __init__.")

        self.mlforecast.fit(
            df_panel,
            id_col=id_col,
            time_col=time_col,
            target_col=target_col,
            static_features=static_features,
        )

    def fit(
        self,
        transactions: List[List[str]],
        df_panel: pd.DataFrame,
        control_cols: List[str],
        id_col: str,
        time_col: str,
        target_col: str,
        static_features: Optional[List[str]] = None,
    ):
        """
        Executa o pipeline completo: Máscara -> Causal DML -> MLForecast.
        """
        self.id_col_ = id_col
        self.time_col_ = time_col
        self.target_col_ = target_col

        # 1. Gera matriz de concorrência
        self.fit_mask_(transactions)

        # 2. Treina elasticidades cruzadas para todos os SKUs que possuem concorrentes
        for sku in self.skus_:
            if sku in self.competitors_dict_:
                self.fit_cross_elasticity_(
                    df_panel=df_panel,
                    id_col=id_col,
                    target_col=target_col,
                    sku_target=sku,
                    control_cols=control_cols,
                )

        # 3. Treina o forecast base de séries temporais
        self.fit_mlforecast_(
            df_panel=df_panel,
            id_col=id_col,
            time_col=time_col,
            target_col=target_col,
            static_features=static_features,
        )

        return self

    def predict(self, h: int, delta_prices: Optional[Dict[str, Dict[str, float]]] = None) -> pd.DataFrame:
        """
        Faz a previsão de demanda futura e aplica o ajuste contrafactual.
        
        delta_prices: Dicionário no formato:
            {'SKU_B': {'SKU_A': 0.15}} -> Representa aumento de 15% no preço do SKU_A afetando o SKU_B.
        """
        # Previsão base não-restrita via Nixtla
        base_forecast = self.mlforecast.predict(h=h)
        adjusted_forecast = base_forecast.copy()

        if delta_prices is None:
            return adjusted_forecast

        # Aplica a correção de canibalização/substituição por SKU
        for target_sku, comp_deltas in delta_prices.items():
            if target_sku not in self.cross_elasticities_:
                continue

            multiplier = 0.0
            for comp_sku, delta_p in comp_deltas.items():
                elasticity = self.cross_elasticities_[target_sku].get(comp_sku, 0.0)
                multiplier += elasticity * delta_p

            # Aplica o ajuste proporcional nas colunas de modelo do MLForecast
            mask_sku = adjusted_forecast[self.id_col_] == target_sku
            model_cols = [
                c for c in adjusted_forecast.columns
                if c not in [self.id_col_, self.time_col_]
            ]
            
            for col in model_cols:
                adjusted_forecast.loc[mask_sku, col] = (
                    adjusted_forecast.loc[mask_sku, col] * (1 + multiplier)
                )

        return adjusted_forecast

In [5]:
fp = ForecastPrice()

In [ ]:
fp.fit()